In [1]:
import pandas as pd
from pathlib import Path

# Ordner mit unseren Kursdateien
ordner = Path.home() / "Desktop" / "Modul 5" / "Aufgaben" / "Woche_1"

# Rohdaten laden
df = pd.read_csv(ordner / "bestellungen_maerz_roh.csv")

# Ausgangszustand anzeigen
print("Zeilen und Spalten:", df.shape)
print("\nSpalten und Datentypen:")
print(df.dtypes)

display(df.head(10))

Zeilen und Spalten: (60, 7)

Spalten und Datentypen:
bestell_id     int64
datum            str
kunde_email      str
land             str
produkt          str
menge          int64
preis_eur        str
dtype: object


,bestell_id,datum,kunde_email,land,produkt,menge,preis_eur
0,2032,2026-03-20,joerg.schaefer@example.com,AT,Dockingstation,4,"129,00"
1,2021,15.03.2026,max.weiss@example.com,DE,Maus,2,19.90
2,2048,2026-03-28,timo.jaeger@example.com,DE,Laptopständer,2,24.95
3,2018,2026-03-14,lena.krueger@example.com,DE,Webcam HD,2,"49,50"
4,2011,2026-03-10,max.weiss@example.com,de,Headset,4,"59,90"
5,2045,27.03.2026,lena.krueger@example.com,DE,Maus,4,19.90
6,2035,2026-03-22,sophie.gross@example.com,DE,Laptopständer,4,"24,95"
7,2016,2026-03-12,NaN,DE,Webcam HD,3,49.50
8,2022,2026-03-15,tom.keller@example.com,DE,Maus,2,"19,90"
9,2031,2026-03-20,timo.jaeger@example.com,AT,Laptopständer,1,24.95


In [2]:
# Ausgangsdaten unverändert sichern
df_vorher = df.copy()

print("1. Fehlende Werte:")
print(df.isna().sum())

print("\n2. Vollständig doppelte Zeilen:")
print(df.duplicated().sum())

print("\n3. Unterschiedliche Länderbezeichnungen:")
print(df["land"].value_counts(dropna=False))

print("\n4. Unterschiedliche Datumsformate:")
print(df["datum"].head(15).to_string(index=False))

print("\n5. Unterschiedliche Preisformate:")
print(df["preis_eur"].head(15).to_string(index=False))

print("\nZusatzkontrolle – ungültige Mengen:")
print("Menge <= 0:", (df["menge"] <= 0).sum())

1. Fehlende Werte:
bestell_id     0
datum          0
kunde_email    5
land           0
produkt        0
menge          0
preis_eur      0
dtype: int64

2. Vollständig doppelte Zeilen:
7

3. Unterschiedliche Länderbezeichnungen:
land
DE             33
AT             12
Deutschland     6
de              5
Österreich      3
at              1
Name: count, dtype: int64

4. Unterschiedliche Datumsformate:
2026-03-20
15.03.2026
2026-03-28
2026-03-14
2026-03-10
27.03.2026
2026-03-22
2026-03-12
2026-03-15
2026-03-20
22.03.2026
2026-03-27
11.03.2026
2026-03-12
2026-03-19

5. Unterschiedliche Preisformate:
129,00
 19.90
 24.95
 49,50
 59,90
 19.90
 24,95
 49.50
 19,90
 24.95
 19.90
129.00
  9.95
 39,90
  9,95

Zusatzkontrolle – ungültige Mengen:
Menge <= 0: 0


In [3]:
# Arbeitskopie erstellen
df_clean = df_vorher.copy()

# 1. Fehlende E-Mails:
# Keine erfundenen Werte eintragen.
# Fehlende Werte bleiben als NaN erhalten.
print("Fehlende E-Mails:", df_clean["kunde_email"].isna().sum())

# 2. Vollständig doppelte Zeilen entfernen
zeilen_vorher = len(df_clean)

df_clean = df_clean.drop_duplicates().copy()

print("Entfernte Duplikate:", zeilen_vorher - len(df_clean))
print("Zeilen nach Bereinigung:", len(df_clean))

# Kontrolle
assert df_clean.duplicated().sum() == 0

Fehlende E-Mails: 5
Entfernte Duplikate: 7
Zeilen nach Bereinigung: 53


In [4]:
# Länderbezeichnungen vor der Bereinigung
print("Vorher:")
print(df_clean["land"].value_counts())

# Bekannte Schreibweisen vereinheitlichen
land_mapping = {
    "Deutschland": "DE",
    "de": "DE",
    "DE": "DE",
    "Österreich": "AT",
    "at": "AT",
    "AT": "AT"
}

df_clean["land"] = df_clean["land"].replace(land_mapping)

# Ergebnis kontrollieren
print("\nNachher:")
print(df_clean["land"].value_counts())

assert set(df_clean["land"].dropna().unique()) == {"DE", "AT"}

Vorher:
land
DE             28
AT             10
Deutschland     6
de              5
Österreich      3
at              1
Name: count, dtype: int64

Nachher:
land
DE    39
AT    14
Name: count, dtype: int64


In [5]:
# Datumswerte vor der Bereinigung sichern
datum_vorher = df_clean["datum"].copy()

# Beide bekannten Formate ausdrücklich erkennen
ist_deutsch = datum_vorher.str.contains(".", regex=False)

datum_deutsch = pd.to_datetime(
    datum_vorher.where(ist_deutsch),
    format="%d.%m.%Y",
    errors="coerce"
)

datum_iso = pd.to_datetime(
    datum_vorher.where(~ist_deutsch),
    format="%Y-%m-%d",
    errors="coerce"
)

# Beide Ergebnisse zusammenführen
df_clean["datum"] = datum_deutsch.fillna(datum_iso)

# Kontrolle
print("Ungültige Datumswerte:", df_clean["datum"].isna().sum())
print("Frühestes Datum:", df_clean["datum"].min())
print("Spätestes Datum:", df_clean["datum"].max())

display(df_clean[["bestell_id", "datum"]].head(10))

assert df_clean["datum"].notna().all()

Ungültige Datumswerte: 0
Frühestes Datum: 2026-03-02 00:00:00
Spätestes Datum: 2026-03-31 00:00:00


,bestell_id,datum
0,2032,2026-03-20
1,2021,2026-03-15
2,2048,2026-03-28
3,2018,2026-03-14
4,2011,2026-03-10
5,2045,2026-03-27
6,2035,2026-03-22
7,2016,2026-03-12
8,2022,2026-03-15
9,2031,2026-03-20


In [6]:
# Originalpreise für den Vorher-Nachher-Vergleich sichern
preise_vorher = df_clean["preis_eur"].copy()

# Komma durch Punkt ersetzen und in Zahlen umwandeln
df_clean["preis_eur"] = pd.to_numeric(
    df_clean["preis_eur"].str.replace(",", ".", regex=False),
    errors="coerce"
)

# Ergebnis überprüfen
print("Datentyp:", df_clean["preis_eur"].dtype)
print("Ungültige Preise:", df_clean["preis_eur"].isna().sum())
print("Preise <= 0:", (df_clean["preis_eur"] <= 0).sum())

print("\nVorher:")
print(preise_vorher.head(10).to_string(index=False))

print("\nNachher:")
print(df_clean["preis_eur"].head(10).to_string(index=False))

# Automatische Qualitätskontrolle
assert df_clean["preis_eur"].notna().all()
assert (df_clean["preis_eur"] > 0).all()

Datentyp: float64
Ungültige Preise: 0
Preise <= 0: 0

Vorher:
129,00
 19.90
 24.95
 49,50
 59,90
 19.90
 24,95
 49.50
 19,90
 24.95

Nachher:
129.00
 19.90
 24.95
 49.50
 59.90
 19.90
 24.95
 49.50
 19.90
 24.95


In [7]:
# Preise der Rohdaten für einen fairen Vergleich umwandeln
df_vergleich = df_vorher.copy()

df_vergleich["preis_eur"] = pd.to_numeric(
    df_vergleich["preis_eur"].str.replace(",", ".", regex=False),
    errors="coerce"
)

# Umsatz je Bestellposition berechnen
df_vergleich["umsatz"] = (
    df_vergleich["menge"] * df_vergleich["preis_eur"]
)

df_clean["umsatz"] = (
    df_clean["menge"] * df_clean["preis_eur"]
)

# Vorher-Nachher-Vergleich
print("=== VORHER / NACHHER ===")

print("\nZeilenanzahl:")
print("Vorher:", len(df_vergleich))
print("Nachher:", len(df_clean))

print("\nUmsatzsumme:")
print("Vorher:", round(df_vergleich["umsatz"].sum(), 2), "€")
print("Nachher:", round(df_clean["umsatz"].sum(), 2), "€")

print("\nLändergruppen vorher:")
print(df_vergleich["land"].value_counts())

print("\nLändergruppen nachher:")
print(df_clean["land"].value_counts())

# Qualitätskontrollen
assert len(df_clean) == 53
assert df_clean.duplicated(subset=df_vorher.columns.tolist()).sum() == 0
assert df_clean["umsatz"].notna().all()

=== VORHER / NACHHER ===

Zeilenanzahl:
Vorher: 60
Nachher: 53

Umsatzsumme:
Vorher: 8054.4 €
Nachher: 7026.3 €

Ländergruppen vorher:
land
DE             33
AT             12
Deutschland     6
de              5
Österreich      3
at              1
Name: count, dtype: int64

Ländergruppen nachher:
land
DE    39
AT    14
Name: count, dtype: int64


In [8]:
# Speicherort festlegen
datei = ordner / "bestellungen_clean.parquet"

# Zeilennummern vereinheitlichen
df_clean = df_clean.reset_index(drop=True)
# Zeilennummern vereinheitlichen
df_clean = df_clean.reset_index(drop=True)

# Bereinigte Daten erneut speichern
df_clean.to_parquet(datei, index=False)

# Datei erneut einlesen
df_test = pd.read_parquet(datei)

# Vollständiger Vergleich
pd.testing.assert_frame_equal(df_clean, df_test)

print("✅ Rücklesetest erfolgreich!")
print("Zeilen:", len(df_test))
print("Spalten:", len(df_test.columns))

display(df_test.head())

✅ Rücklesetest erfolgreich!
Zeilen: 53
Spalten: 8


,bestell_id,datum,kunde_email,land,produkt,menge,preis_eur,umsatz
0,2032,2026-03-20,joerg.schaefer@example.com,AT,Dockingstation,4,129.00,516.0
1,2021,2026-03-15,max.weiss@example.com,DE,Maus,2,19.90,39.8
2,2048,2026-03-28,timo.jaeger@example.com,DE,Laptopständer,2,24.95,49.9
3,2018,2026-03-14,lena.krueger@example.com,DE,Webcam HD,2,49.50,99.0
4,2011,2026-03-10,max.weiss@example.com,DE,Headset,4,59.90,239.6


## Woche 1 – Tag 5: Datenbereinigung und Qualitätskontrolle

### 1. Ausgangslage

Die Datei `bestellungen_maerz_roh.csv` enthielt 60 Datensätze mit 7 Spalten. Bei der Untersuchung wurden fünf Datenqualitätsprobleme festgestellt.

### 2. Bereinigungsregeln und Begründungen

**Befund 1: Fehlende E-Mail-Adressen**
- Befund: 5 fehlende Werte in `kunde_email`.
- Regel: Fehlende Werte bleiben als `NaN` erhalten.
- Begründung: E-Mail-Adressen dürfen nicht erfunden werden.
- Beleg: Die fehlenden Werte wurden mit `isna().sum()` nachgewiesen und bewusst nicht verändert.

**Befund 2: Doppelte Datensätze**
- Befund: 7 vollständig doppelte Zeilen.
- Regel: Nur vollständig identische Datensätze entfernen.
- Begründung: Doppelte Zeilen können Umsatzberechnungen verfälschen.
- Beleg: Die Zeilenanzahl sank von 60 auf 53. Eine Prüfung mit `duplicated()` bestätigte, dass keine vollständigen Duplikate übrig blieben.

**Befund 3: Uneinheitliche Länderbezeichnungen**
- Befund: 6 unterschiedliche Schreibweisen für Deutschland und Österreich.
- Regel: Alle bekannten Varianten auf `DE` und `AT` vereinheitlichen.
- Begründung: Gleiche Länder sollen bei Auswertungen nicht getrennt gezählt werden.
- Beleg: Nach der Bereinigung gab es nur noch `DE` (39) und `AT` (14).

**Befund 4: Unterschiedliche Datumsformate**
- Befund: Datumswerte lagen im ISO-Format und im deutschen Format vor.
- Regel: Beide Formate ausdrücklich erkennen und in ein einheitliches Datumsformat umwandeln.
- Begründung: Falsch interpretierte Datumswerte können zeitliche Analysen verfälschen.
- Beleg: Alle 53 Datumswerte wurden erfolgreich umgewandelt. Der Zeitraum reicht vom 02.03.2026 bis zum 31.03.2026.

**Befund 5: Unterschiedliche Preisformate**
- Befund: Preise verwendeten teilweise Kommas und teilweise Punkte als Dezimaltrennzeichen.
- Regel: Dezimalkommas vereinheitlichen und Preise in numerische Werte umwandeln.
- Begründung: Umsatzberechnungen benötigen numerische Werte.
- Beleg: Die Spalte `preis_eur` besitzt jetzt den Datentyp `float64`. Es gibt keine ungültigen oder nicht positiven Preise.

### 3. Vorher-Nachher-Vergleich

| Kennzahl | Vorher | Nachher |
|---|---:|---:|
| Datensätze | 60 | 53 |
| Umsatzsumme | 8.054,40 € | 7.026,30 € |
| Ländergruppen | 6 | 2 |
| Vollständige Duplikate | 7 | 0 |

Die Umsatzsumme sank um 1.028,10 €, weil sieben vollständig doppelte Datensätze entfernt wurden. Da die Datei keine Kategoriespalte enthält, wurden die Ländergruppen als Kategorien verglichen.

### 4. Parquet-Export und Rücklesetest

Die bereinigten Daten wurden als `bestellungen_clean.parquet` gespeichert und anschließend erneut eingelesen.

Der Rücklesetest mit `pd.testing.assert_frame_equal()` war erfolgreich. Alle 53 Datensätze und 8 Spalten wurden korrekt übernommen.

### 5. Copilot-Reflexion

Bei der KI-gestützten Datenbereinigung ist es wichtig, nicht mehr zu verändern, als die tatsächlichen Befunde rechtfertigen. Beispielsweise wäre es problematisch, fehlende E-Mail-Adressen automatisch zu ergänzen oder Bestellungen allein aufgrund gleicher Bestellnummern zu löschen.

Deshalb wurden nur vollständig identische Zeilen entfernt, bekannte Länderbezeichnungen vereinheitlicht und Datums- sowie Preisformate korrigiert. Die fünf fehlenden E-Mail-Adressen blieben bewusst unverändert, da keine verlässlichen Ersatzwerte vorliegen.